In [1]:
import argparse
import torch
import os
import json
from tqdm import tqdm
import shortuuid

# 需要定义llava模块的位置，即指定PATH变量
import sys
llava_module_dir = "/root/userfolder/MIL/VL-MIL"
sys.path.insert(0, llava_module_dir)

from llava.constants import IMAGE_TOKEN_INDEX
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path 

from feature_utils import generate_hidden_states

/root/userfolder/anaconda3/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nfi_desc_path = "/root/commonfile/NFI/jsons/screenings"
model_path = "/root/userfolder/data-ckpts/VL-MIL/checkpoints/v2/llava/merged/llava-onevision-qwen2-7b-si-ft-3epoch-NFD-freeze_backbone_valid_1gpu"
image_folder = "/root/commonfile/InfantVQA"
device='cuda'

In [3]:
nfi_list= []
for split in ['train', 'valid', 'test']:
    json_path = os.path.join(nfi_desc_path, f"nfi_{split}.json")
    assert os.path.exists(json_path)
    with open(json_path, 'r') as f:
        nfi_list.extend(json.load(f))
len(nfi_list)

9260

In [4]:
# 匹配某个文件在文件夹中的序号
def get_image_idx(file:str) -> int:
    ndots = file.count(".")
    if ndots == 2:
        return int(file.split(".")[-2])
    elif ndots == 1:
        substr = file.split(".")[-2]
        s = len(substr) - 1
        while("0" <= substr[s] <= "9" and s >= 0):
            s -= 1
        return int(substr[s + 1:])
    else:
        return None

In [5]:
nfi_list[0]

{'id': 1,
 'num_image': 20,
 'patient': 'NEG/ai_bingeying_1800076_20180111',
 'image': ['nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.1.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.2.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.3.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.4.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.5.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.6.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.7.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.8.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.9.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd

In [6]:
patient = nfi_list[0]
def form_conversation(patient):
    pid = patient['id']
    name = patient['patient'].split("/")[-1]
    img_num = patient['num_image']
    images = patient['image']
    
    patient_sample = []

    for img in images:
        img_idx = get_image_idx(img)
        sample = {
            "id": pid + img_idx,
            "image": img,
            "patient": name,
            "description": None,
            "conversations": [
                {
                    "from": "human",
                    "value": "<image>\nPlease tell me the possible abnormality in the newborn fundus image in Chinese."
                },
                {
                    "from": "gpt",
                    "value": "<think>分析过程。</think> <answer>未知。</answer>."
                }
            ]
        }
    
        patient_sample.append(sample)
    return patient_sample 

nfi_desc_dataset = [form_conversation(p) for p in nfi_list]
print(f"NFI dataset has totally {len(nfi_desc_dataset)} patients.")
len(form_conversation(nfi_list[1]))

NFI dataset has totally 9260 patients.


18

加载模型

In [7]:
from llava.model.multimodal_encoder.siglip_encoder import SigLipVisionModel, SigLipImageProcessor
vision_model = SigLipVisionModel.from_pretrained("/root/userfolder/model_weights/siglip-so400m-patch14-384").half().to(device)
image_processor = SigLipImageProcessor()


In [8]:
siglip_pooler_path = f'/root/commonfile/InfantVQA/nfi/features/siglip_pooler'
os.makedirs(siglip_pooler_path, exist_ok=True)

In [9]:
assert False

AssertionError: 

按照病人进行多图描述

In [10]:
from PIL import Image
from torch.nn.utils.rnn import pad_sequence

pbar = tqdm(nfi_desc_dataset)
for sample in pbar:
    pbar.set_description(sample[0]['patient'])  # 动态更新描述
    pbar.refresh()  # 立即刷新显示
    siglip_pooler_list = []
    # A sample contains multiple conversations turns of multiple images
    for line in sample:
        patient = line['patient']
        image_files = line['image']
        image_tensors = []
        if isinstance(image_files, str):
            image_files = [image_files]
        for image_file in image_files:
            image = Image.open(os.path.join(image_folder, image_file))
            image_tensor = image_processor.preprocess(image, return_tensors='pt')['pixel_values'].cuda()
            # image_tensors.append(image_tensor.half())
            feature = vision_model(image_tensor.half())
            siglip_pooler_list.append(feature.pooler_output.cpu().detach())
            
    
    # save siglip feature
    siglip_features = torch.stack(siglip_pooler_list, dim=0).squeeze(0)
    torch.save(siglip_features.cpu(), os.path.join(siglip_pooler_path, patient + '.pt'))

zu_peixiaoying_1602541_20161116: 100%|██████████| 9260/9260 [2:25:49<00:00,  1.06it/s]          


torch.Size([20, 1, 1152])